# Image only Regression Model
This part implements image only regression models using Deit backbone architecture. The model extracts visual features from input images and predicts land cover composition percentages through a regression head. This serves as a baseline to evaluate how well visual information alone can capture area distribution without any additional semantic input.


In [50]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms, models

from sklearn.metrics import mean_absolute_error, mean_squared_error

In [51]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [52]:
DATA_ROOT = "/content/drive/MyDrive/subset"

In [53]:
import os

print(os.listdir(DATA_ROOT))

['captions_subset.csv', '.DS_Store', 'images']


In [54]:
DATA_ROOT = "/content/drive/MyDrive/subset"
CSV_PATH = os.path.join(DATA_ROOT, "captions_subset.csv")
IMAGE_DIR = os.path.join(DATA_ROOT, "images")

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

BATCH_SIZE = 32
IMG_SIZE = 224
EPOCHS = 7
LR = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [55]:
df = pd.read_csv(CSV_PATH)

print(df.head())
print("Dataset size:", len(df))

     filename  split  Tree  Shrub  Grass  Crop  Built-up  Barren  Water  \
0  267687.png  synth    85     13      2     0         0       0      0   
1  222448.png  synth    18      0     81     0         0       1      0   
2   14328.png  synth     2      0     65     1        22      10      0   
3  224249.png  synth    61      0     39     0         0       0      0   
4  218094.png  synth     0      0      5    93         2       0      0   

                                    hybrid_gemma3-4b  \
0  The image depicts a heavily forested mountaino...   
1  The image depicts a landscape dominated by ext...   
2  The image depicts a landscape dominated by ext...   
3  The image depicts a mountainous region dominat...   
4  The image depicts a landscape dominated by ext...   

                                  hybrid_qwen3-vl-8b  \
0  The scene is dominated by dense tree cover (85...   
1  The scene depicts a predominantly grass-covere...   
2  The scene is dominated by grasslands (65%

In [56]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

print("Train:", len(train_df))
print("Val:", len(val_df))

Train: 400
Val: 100


In [77]:
import pandas as pd

stats = []

for col in TARGET_COLS:
    avg = train_df[col].mean()
    std = train_df[col].std()
    non_zero_ratio = (train_df[col] > 0).mean()

    stats.append({
        "class": col,
        "mean_%": avg,
        "std_%": std,
        "presence_ratio": non_zero_ratio
    })

stats_df = pd.DataFrame(stats).sort_values(by="mean_%", ascending=False)
stats_df

,class,mean_%,std_%,presence_ratio
2,Grass,45.6025,33.890055,0.9450
0,Tree,29.6475,34.917062,0.7525
3,Crop,15.8425,28.934940,0.4550
5,Barren,4.5100,11.712723,0.5275
6,Water,2.1500,9.959693,0.0850
4,Built-up,1.4675,6.552961,0.2475
1,Shrub,0.7800,4.347280,0.0925


In [57]:
class AreaDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(row[TARGET_COLS].values.astype(np.float32))

        return image, target

In [58]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = AreaDataset(train_df, IMAGE_DIR, train_transform)
val_ds   = AreaDataset(val_df, IMAGE_DIR, val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

Images are resized to 224×224, randomly flipped during training, converted to tensors, and normalized using ImageNet statistics. This ensures compatibility with the pretrained DeiT model. The final layer is replaced to output 7 values, and a softmax scaled by 100 is applied to obtain percentage predictions.

Deit image

In [59]:
!pip -q install timm

In [60]:
import timm
import torch
import torch.nn as nn
from tqdm import tqdm
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [61]:
class DeiTImageRegressor(nn.Module):
    def __init__(self, num_outputs=7):
        super().__init__()

        self.backbone = timm.create_model("deit_tiny_patch16_224", pretrained=True)
        in_features = self.backbone.head.in_features
        self.backbone.head = nn.Linear(in_features, num_outputs)

    def forward(self, x):
        logits = self.backbone(x)
        preds = torch.softmax(logits, dim=1) * 100.0
        return preds

In [62]:
model = DeiTImageRegressor(num_outputs=len(TARGET_COLS)).to(DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

In [63]:
def train_epoch_image_only():
    model.train()
    total_loss = 0.0

    loop = tqdm(train_loader, desc="Training")

    for imgs, targets in loop:
        imgs = imgs.to(DEVICE)
        targets = targets.to(DEVICE)

        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        loop.set_postfix(loss=loss.item())

    return total_loss / len(train_loader.dataset)


def evaluate_image_only():
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for imgs, targets in tqdm(val_loader, desc="Validation"):
            imgs = imgs.to(DEVICE)
            preds = model(imgs).cpu().numpy()

            all_preds.append(preds)
            all_targets.append(targets.numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    per_class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return mae, rmse, per_class_mae, y_true, y_pred

In [64]:
best_mae = float("inf")
best_state = None

for epoch in range(EPOCHS):
    loss = train_epoch_image_only()
    mae, rmse, _, _, _ = evaluate_image_only()

    print(f"Epoch {epoch+1}")
    print(f"  Loss: {loss:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")

    if mae < best_mae:
        best_mae = mae
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print("  --> saved best model")

Validation: 100%|██████████| 4/4 [00:00<00:00,  5.02it/s]


Epoch 1
  Loss: 412.9970
  MAE:  6.0810
  RMSE: 11.8717
  --> saved best model


Validation: 100%|██████████| 4/4 [00:00<00:00,  5.02it/s]


Epoch 2
  Loss: 161.2640
  MAE:  5.5554
  RMSE: 10.8452
  --> saved best model


Validation: 100%|██████████| 4/4 [00:00<00:00,  4.94it/s]


...

Epoch 5
  Loss: 76.2127
  MAE:  5.3235
  RMSE: 9.8463


Validation: 100%|██████████| 4/4 [00:00<00:00,  5.15it/s]


Epoch 6
  Loss: 57.4524
  MAE:  5.0536
  RMSE: 10.1181


Validation: 100%|██████████| 4/4 [00:00<00:00,  5.33it/s]

Epoch 7
  Loss: 40.3810
  MAE:  4.6858
  RMSE: 9.2434
  --> saved best model


In [65]:
model.load_state_dict(best_state)

mae, rmse, per_class_mae, y_true, y_pred = evaluate_image_only()

print("Best Val MAE:", mae)
print("Best Val RMSE:", rmse)

for col, score in zip(TARGET_COLS, per_class_mae):
    print(f"{col}: {score:.2f}")

Validation: 100%|██████████| 4/4 [00:00<00:00,  5.32it/s]

Best Val MAE: 4.685823917388916
Best Val RMSE: 9.243393471879067
Tree: 5.79
Shrub: 0.85
Grass: 11.57
Crop: 7.73
Built-up: 1.30
Barren: 4.09
Water: 1.47


# Deit with text
This section extends the baseline by incorporating textual descriptions alongside images. Image features are combined with text embeddings obtained from a pretrained language model, and the fused representation is used for regression. This setup enables the model to leverage both visual and semantic information, allowing evaluation of the impact of multimodal learning on area estimation performance.

In [68]:
from transformers import AutoModel

class DeiTTextConcatRegressor(nn.Module):
    def __init__(self, num_outputs=7):
        super().__init__()

        # DeiT image encoder
        self.image_backbone = timm.create_model("deit_tiny_patch16_224", pretrained=True)
        img_dim = self.image_backbone.head.in_features
        self.image_backbone.head = nn.Identity()
        self.image_proj = nn.Linear(img_dim, 256)

        # DistilBERT text encoder
        self.text_backbone = AutoModel.from_pretrained("distilbert-base-uncased")
        text_dim = self.text_backbone.config.hidden_size
        self.text_proj = nn.Linear(text_dim, 256)

        # Fusion head
        self.regressor = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_outputs)
        )

    def forward(self, image, input_ids, attention_mask):
        # image
        img_feat = self.image_backbone(image)
        img_feat = self.image_proj(img_feat)

        # text
        text_out = self.text_backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_feat = text_out.last_hidden_state[:, 0, :]
        text_feat = self.text_proj(text_feat)

        # concat fusion
        fused = torch.cat([img_feat, text_feat], dim=1)

        logits = self.regressor(fused)
        preds = torch.softmax(logits, dim=1) * 100.0
        return preds

In [69]:
from transformers import AutoTokenizer

TEXT_COL = "vision_qwen3-vl-8b"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

Among the available caption sources, the vision generated caption was used because it provides a natural textual summary of the image content and is  aligned with the visual input. This does not constitute data leakage, because the caption is derived only from the image itself and does not use the ground truth area percentages or segmentation labels as input. In other words, the model receives additional semantic information extracted from the same image, but not target information.

In [70]:
class MultiModalAreaDataset(Dataset):
    def __init__(self, df, image_dir, text_col, target_cols, tokenizer, max_len=128, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.text_col = text_col
        self.target_cols = target_cols
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        text = str(row[self.text_col])
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        target = torch.tensor(row[self.target_cols].values.astype(np.float32), dtype=torch.float32)

        return {
            "image": image,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "target": target
        }

In [71]:
train_ds = MultiModalAreaDataset(
    train_df, IMAGE_DIR, TEXT_COL, TARGET_COLS, tokenizer, max_len=MAX_LEN, transform=train_transform
)

val_ds = MultiModalAreaDataset(
    val_df, IMAGE_DIR, TEXT_COL, TARGET_COLS, tokenizer, max_len=MAX_LEN, transform=val_transform
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

Image preprocessing is the same as the image only setup. Text captions are tokenized and processed using DistilBERT, producing text embeddings. Image and text features are projected to a common space and concatenated before regression. The final output is normalized with softmax to represent class percentages.

In [72]:
model = DeiTTextConcatRegressor(num_outputs=len(TARGET_COLS)).to(DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [73]:
def train_epoch_multimodal():
    model.train()
    total_loss = 0.0

    loop = tqdm(train_loader, desc="Training")

    for batch in loop:
        images = batch["image"].to(DEVICE)
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        targets = batch["target"].to(DEVICE)

        optimizer.zero_grad()
        preds = model(images, input_ids, attention_mask)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        loop.set_postfix(loss=loss.item())

    return total_loss / len(train_loader.dataset)


def evaluate_multimodal():
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            images = batch["image"].to(DEVICE)
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            targets = batch["target"]

            preds = model(images, input_ids, attention_mask).cpu().numpy()

            all_preds.append(preds)
            all_targets.append(targets.numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    per_class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return mae, rmse, per_class_mae, y_true, y_pred

In [75]:
best_mae = float("inf")
best_state = None

for epoch in range(EPOCHS):
    loss = train_epoch_multimodal()
    mae, rmse, _, _, _ = evaluate_multimodal()

    print(f"Epoch {epoch+1}")
    print(f"  Loss: {loss:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")

    if mae < best_mae:
        best_mae = mae
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print("  --> saved best model")

Validation: 100%|██████████| 4/4 [00:00<00:00,  4.48it/s]


Epoch 1
  Loss: 53.2885
  MAE:  4.7916
  RMSE: 8.9888
  --> saved best model


Validation: 100%|██████████| 4/4 [00:00<00:00,  4.40it/s]


Epoch 2
  Loss: 47.8939
  MAE:  5.0784
  RMSE: 9.9815


Validation: 100%|██████████| 4/4 [00:00<00:00,  4.52it/s]


...

Epoch 5
  Loss: 32.5427
  MAE:  4.5468
  RMSE: 9.3238
  --> saved best model


Validation: 100%|██████████| 4/4 [00:00<00:00,  4.60it/s]


Epoch 6
  Loss: 30.2825
  MAE:  4.2239
  RMSE: 8.0337
  --> saved best model


Validation: 100%|██████████| 4/4 [00:00<00:00,  4.59it/s]

Epoch 7
  Loss: 26.4720
  MAE:  4.1297
  RMSE: 8.1662
  --> saved best model


In [76]:
model.load_state_dict(best_state)

mae, rmse, per_class_mae, y_true, y_pred = evaluate_multimodal()

print("Best Val MAE:", mae)
print("Best Val RMSE:", rmse)

for col, score in zip(TARGET_COLS, per_class_mae):
    print(f"{col}: {score:.2f}")

Validation: 100%|██████████| 4/4 [00:00<00:00,  4.36it/s]

Best Val MAE: 4.1297454833984375
Best Val RMSE: 8.1661597373525
Tree: 5.06
Shrub: 1.55
Grass: 9.82
Crop: 7.10
Built-up: 1.08
Barren: 3.04
Water: 1.26


The DeiT based models show a clear improvement when incorporating textual information. The image only model achieves a validation MAE of 4.69 and RMSE of 9.24, while the multimodal model improves to 4.13 MAE and 8.16 RMSE. This indicates that textual descriptions provide useful complementary information for estimating land cover composition under limited data.

At the class level, improvements are consistent across most categories. Notably, Grass (MAE 11.57 to 9.82) and Crop (MAE 7.73 to 7.10) show clear error reductions, suggesting that text helps capture semantic distinctions in vegetationNrelated classes. Similarly, Barren (MAE 4.09 to 3.04) and Water (MAE 1.47 to 1.26) also benefit from multimodal input. Smaller improvements are observed in Tree (MAE 5.79 to 5.06) too.

Overall, these results demonstrate that adding text enhances performance for DeiT, particularly for classes with more complex or ambiguous visual patterns, supporting the effectiveness of multimodal learning under data constraints.